In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

from utils import Vcom, get_comoving_distance_h, c, z_min
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
import copy

# from Ng8_func import Ng8_percentile_wprp, Ng8_percentile_wprp_theory, return_box_rsd_pos
from Ng8_func import N_g_8, N_g_8_theory, rand_z_assign, select_top20_in_random, select_bottom20_in_random, cross_correlation_wprp, cross_correlation_wprp_theory, return_box_rsd_pos

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_{\rm r}: [-22,-21]$',r'$M_{\rm r}: [-21,-20]$',r'$M_{\rm r}: [-20,-19]$',r'$M_{\rm r}: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

In [ ]:
M_thr_arr_DDP  = np.array((-20.0, -20.0, -19.0, -18.0))
z_max_arr_DDP  = np.array((31900, 31900, 19250, 12500))/ c

In [ ]:
### Comoving pi_max and wp bins in Mpc/h
rp_bins = np.logspace(-0.8, 1.6, 13)
rp_bins_x = (rp_bins[1:] + rp_bins[:-1]) / 2


from matplotlib import cm

colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]

In [ ]:
def Ng8_percentile_wprp_DDP(gal, random, Ng8_thr, pi_max, rp_bins, nthread = 55):
    
    D1_ra  = copy.deepcopy(gal[:, 0])
    D1_dec = copy.deepcopy(gal[:, 1])
    D1_z   = copy.deepcopy(gal[:, 2])
    R1_ra  = copy.deepcopy(random[:, 0])
    R1_dec = copy.deepcopy(random[:, 1])
    R1_z   = copy.deepcopy(random[:, 2])

    # len(Ng8_thr) = len(D1_ra)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)

    D2_ra  = select_bottom20_in_random(D1_ra , Ng8_thr)
    D2_dec = select_bottom20_in_random(D1_dec, Ng8_thr)
    D2_z   = select_bottom20_in_random(D1_z  , Ng8_thr)

    R2_ra  = R1_ra
    R2_dec = R1_dec
    R2_z   = rand_z_assign(R1_z, D2_z) # shuffle z from D2_z(thr masked)

    wprp_bottom20 = cross_correlation_wprp( D1_ra, D1_dec, D1_z, R1_ra, R1_dec, R1_z, 
                                            D2_ra, D2_dec, D2_z, R2_ra, R2_dec, R2_z,
                                            pi_max, rp_bins, gal_weights = 1.0, random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 top 20%)

    D2_ra  = select_top20_in_random(D1_ra , Ng8_thr)
    D2_dec = select_top20_in_random(D1_dec, Ng8_thr)
    D2_z   = select_top20_in_random(D1_z  , Ng8_thr)

    R2_ra  = R1_ra
    R2_dec = R1_dec
    R2_z   = rand_z_assign(R1_z, D2_z) # shuffle z from D2_z(thr masked)

    wprp_top20 = cross_correlation_wprp( D1_ra, D1_dec, D1_z, R1_ra, R1_dec, R1_z, 
                                            D2_ra, D2_dec, D2_z, R2_ra, R2_dec, R2_z,
                                            pi_max, rp_bins, gal_weights = 1.0, random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    return wprp_bottom20, wprp_top20


In [ ]:
def Ng8_percentile_wprp_theory_DDP(gal, random, Ng8_thr, pi_max, rp_bins, boxsize, nthread = 55):
    D1_X = copy.deepcopy(gal[:, 0])
    D1_Y = copy.deepcopy(gal[:, 1])
    D1_Z = copy.deepcopy(gal[:, 2])
    R1_X = copy.deepcopy(random[:, 0])
    R1_Y = copy.deepcopy(random[:, 1])
    R1_Z = copy.deepcopy(random[:, 2])

 
    # len(Ng8_thr) = len(D1_ra)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)

    D2_X = select_bottom20_in_random(D1_X, Ng8_thr)
    D2_Y = select_bottom20_in_random(D1_Y, Ng8_thr)
    D2_Z = select_bottom20_in_random(D1_Z, Ng8_thr)

    R2_X = R1_X
    R2_Y = R1_Y
    R2_Z = R1_Z

    wprp_bottom20 = cross_correlation_wprp_theory(  D1_X, D1_Y, D1_Z, R1_X, R1_Y, R1_Z, 
                                                    D2_X, D2_Y, D2_Z, R2_X, R2_Y, R2_Z,
                                                    pi_max, rp_bins, boxsize, gal_weights  = 1.0, 
                                                    random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 top 20%)

    D2_X = select_top20_in_random(D1_X, Ng8_thr)
    D2_Y = select_top20_in_random(D1_Y, Ng8_thr)
    D2_Z = select_top20_in_random(D1_Z, Ng8_thr)

    R2_X = R1_X
    R2_Y = R1_Y
    R2_Z = R1_Z

    wprp_top20 = cross_correlation_wprp_theory( D1_X, D1_Y, D1_Z, R1_X, R1_Y, R1_Z, 
                                                D2_X, D2_Y, D2_Z, R2_X, R2_Y, R2_Z,
                                                pi_max, rp_bins, boxsize, gal_weights = 1.0, 
                                                random_weights = 1.0, nthread = nthread)
    ######################################################################################################################
    return wprp_bottom20, wprp_top20

#### Mr_C_bins(~2h each)

In [ ]:
for j in range(4):
    print(M_thr_arr[j],M_thr_arr[j+1], '|', M_thr_arr_DDP[j],  '|', z_max_arr_DDP[j])


#### CCMD 32 quads & SDSS bright1

In [ ]:
data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + 'Observer_1' + '_' + 'Quad0' + '_processed_catalog.fits')[1].data

data

In [ ]:
### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

quads = ['bright1']
vars  = ['SDSS']

for var in vars:
    for quad in quads:
        # data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
        data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
        N_total = len(data) # 获取母表总长度

        output_dict1 = collections.defaultdict(list)
        output_dict2 = collections.defaultdict(list)
    #########################################################################################################
        for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
            ### DDP ###################################################################################
            mask_DDP = np.logical_and(  data['Mr'] < M_thr_arr_DDP[j],
                                        np.logical_and(z_min <= data['z'], data['z'] < z_max_arr_DDP[j] ) )
            data_DDP = data[mask_DDP]
            pos_DDP  = np.array([   np.array(data_DDP['ra']),
                                    np.array(data_DDP['dec']),    
                                    np.array(data_DDP['z'])]).T
            Ng8_DDP  = N_g_8(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

            # === 高效映射核心步骤 ===
            # 创建一个与原始 data 等长的数组，初始化为 NaN
            Ng8_full = np.full(N_total, np.nan)
            # 将算好的 Ng8 填入 DDP 星系所在的位置
            Ng8_full[mask_DDP] = Ng8_DDP
            # ========================
            ###########################################################################################
            for i in range(4):
                stat_bot = []
                stat_top = []
                ### Mr mask is related to the z_max 
                mask = np.logical_and(  np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                        np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                        np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
                if j == 3:
                    pi_max = 40.
                else :
                    pi_max = 60.

                data_thr = data[mask]
                # === 高效提取对应星系的 Ng8 ===
                # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
                Ng8_thr  = Ng8_full[mask]
                
                # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
                # 如果你的物理逻辑保证了 thr 样本一定是 DDP 的子集，这一步不会剔除任何星系
                valid_idx = ~np.isnan(Ng8_thr)
                data_thr_valid = data_thr[valid_idx]
                Ng8_thr_valid  = Ng8_thr[valid_idx]
                # ==============================

                coords_gal_thr  = np.array([np.array(data_thr_valid['ra']),
                                            np.array(data_thr_valid['dec']),    
                                            np.array(data_thr_valid['z'])]).T
                
                # coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_wp_randoms/CCMD_'
                                        #    + var + '_' + quad + '_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms.npz')['randoms']
                coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1/SDSS_randoms_' 
                                          + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']

                wprp_bottom20, wprp_top20 = Ng8_percentile_wprp_DDP(coords_gal_thr, coords_rand_thr, Ng8_thr_valid, pi_max, rp_bins, )
                stat_bot.append(wprp_bottom20)
                stat_top.append(wprp_top20)

                output_dict1['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_bot for item in sublist])
                output_dict2['wprp_top20_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_top for item in sublist])

        # np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz', **output_dict1)
        # np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz', **output_dict2)
        np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz', **output_dict1)
        np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz', **output_dict2)

#### SDSS bright1 jackknife cov

In [ ]:
# ### SDSS bright1 cov
# n_jk = 100

# data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
# N_total = len(data) # 获取母表总长度

# output_dict1    = collections.defaultdict(list)
# output_dict2    = collections.defaultdict(list)

# output_dict1_jk = collections.defaultdict(list)
# output_dict2_jk = collections.defaultdict(list)

# #########################################################################################################
# for j in range(4): # Mr bins
#     print('###', Mr_thr_text[j], '############')
#     ### DDP ###################################################################################
#     mask_DDP = np.logical_and(  data['Mr'] < M_thr_arr_DDP[j],
#                                 np.logical_and(z_min <= data['z'], data['z'] < z_max_arr_DDP[j] ) )
#     data_DDP = data[mask_DDP]
#     pos_DDP  = np.array([   np.array(data_DDP['ra']),
#                             np.array(data_DDP['dec']),    
#                             np.array(data_DDP['z'])]).T
#     Ng8_DDP  = N_g_8(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

#     # === 高效映射核心步骤 ===
#     # 创建一个与原始 data 等长的数组，初始化为 NaN
#     Ng8_full = np.full(N_total, np.nan)
#     # 将算好的 Ng8 填入 DDP 星系所在的位置
#     Ng8_full[mask_DDP] = Ng8_DDP
#     # ========================
#     ###########################################################################################
#     for i in range(4): # color bins
#         print('###', Co_thr_text[j,i], '############')
#         stat_jk1 = np.zeros([n_jk, 1 + 12])
#         stat_jk2 = np.zeros([n_jk, 1 + 12])
#         ### Mr mask is related to the z_max 
#         mask = np.logical_and(np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
#                                 np.logical_and(np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
#                                                 np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
#         if j == 3:
#             pi_max = 40
#         else:
#             pi_max = 60

#         data_thr = data[mask]
#         # === 高效提取对应星系的 Ng8 ===
#         # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
#         Ng8_thr  = Ng8_full[mask]
        
#         # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
#         # 如果你的物理逻辑保证了 thr 样本一定是 DDP 的子集，这一步不会剔除任何星系
#         valid_idx = ~np.isnan(Ng8_thr)
#         data_thr_valid = data_thr[valid_idx]
#         Ng8_thr_valid  = Ng8_thr[valid_idx]
#         # ==============================

#         coords_gal_thr  = np.array([np.array(data_thr_valid['ra']),
#                                     np.array(data_thr_valid['dec']),    
#                                     np.array(data_thr_valid['z'])]).T
        
#         path0 = '/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1/'
#         coords_rand_thr = np.load(path0 + 'SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
#         labels_rand_thr = np.load(path0 + 'SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['jk_label']
    
#         for k in range(n_jk):
#             if k % 10 == 0:
#                 print(f"第 {k} 次循环")
#             stat_jk1[k, 0] = np.sum(data_thr_valid['jk_label'] != k) / (Vcom(z_max_arr[j])
#                                 * np.sum(labels_rand_thr != k) / len(labels_rand_thr))
#             stat_jk2[k, 0] = np.sum(data_thr_valid['jk_label'] != k) / (Vcom(z_max_arr[j])
#                                 * np.sum(labels_rand_thr != k) / len(labels_rand_thr))
            
#             wprp_bottom20, wprp_top20 = Ng8_percentile_wprp_DDP(coords_gal_thr[data_thr_valid['jk_label'] != k],
#                                                                 coords_rand_thr[labels_rand_thr != k],
#                                                                 Ng8_thr_valid[data_thr_valid['jk_label'] != k],
#                                                                 pi_max, rp_bins)
#             stat_jk1[k, 1:13] = wprp_bottom20
#             stat_jk2[k, 1:13] = wprp_top20

#         output_dict1['wprp_bottom20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.cov(stat_jk1.T, bias = 1) * (n_jk - 1)
#         output_dict2['wprp_top20_cov'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.cov(stat_jk2.T, bias = 1) * (n_jk - 1)
        
#         output_dict1_jk['wprp_bottom20_jk_sample_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = stat_jk1
#         output_dict2_jk['wprp_top20_jk_sample_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = stat_jk2
    

# np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_cov_pimax60_60_60_40.npz', **output_dict1)
# np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_cov_pimax60_60_60_40.npz', **output_dict2)

# np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_cov_jkknife_sample_pimax60_60_60_40.npz', **output_dict1_jk)
# np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_cov_jkknife_sample_pimax60_60_60_40.npz', **output_dict2_jk)

#### CCMD 32 quads cov


In [ ]:
### CCMD 32 Quads cov
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

output_dict_bottom_stat = collections.defaultdict(list)
output_dict_top_stat    = collections.defaultdict(list)
output_dict_bottom_cov  = collections.defaultdict(list)
output_dict_top_cov     = collections.defaultdict(list)

for j in range(4): # Mr, z
    for i in range(4): # Color
        bottom_data  = []
        top_data     = []
        for var in vars:
            for quad in quads:
                bottom_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
                bottom_data.append(bottom_temp['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

                top_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')
                top_data.append(top_temp['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

        bottom_data  = np.array(bottom_data)
        top_data     = np.array(top_data)     

        bottom_stat = []
        bottom_cov  = []
        top_stat    = []
        top_cov     = []

        original_statistics_bottom = np.mean(bottom_data, axis=0)
        original_statistics_top    = np.mean(top_data    , axis=0)
        bottom_cov_matrix          = np.cov(bottom_data, rowvar=False)  # rowvar=False, column for variants
        top_cov_matrix             = np.cov(top_data, rowvar=False)  # rowvar=False, column for variants
        
        # bottom_cov_matrix          = np.std(bottom_data, axis=0)  # rowvar=False, column for variants
        # top_cov_matrix             = np.std(top_data, axis=0)  # rowvar=False, column for variants

        bottom_stat.append(original_statistics_bottom)
        bottom_cov.append(bottom_cov_matrix)
        top_stat.append(original_statistics_top)
        top_cov.append(top_cov_matrix)

        output_dict_bottom_stat['wprp_bottom20_'     + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in bottom_stat for item in sublist])
        output_dict_bottom_cov[ 'wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in bottom_cov for item in sublist])
        output_dict_top_stat['wprp_top20_'     + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in top_stat for item in sublist])
        output_dict_top_cov[ 'wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in top_cov for item in sublist])

    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_stat_mean.npz', **output_dict_bottom_stat)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_stat_mean.npz', **output_dict_top_stat)
    
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_cov.npz', **output_dict_bottom_cov)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_cov.npz', **output_dict_top_cov)

### CCMD main box w/o rsd

In [ ]:
### CCMD main box without rsd
data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
N_total = len(data)

output_dict1 = collections.defaultdict(list)
output_dict2 = collections.defaultdict(list)
#########################################################################################################
for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
        ### DDP ###################################################################################
    mask_DDP = data['Mr'] < M_thr_arr_DDP[j]                                   
    data_DDP = data[mask_DDP]
    pos_DDP  = np.array([   np.array(data_DDP['x'] % 1000),
                            np.array(data_DDP['y'] % 1000),    
                            np.array(data_DDP['z'] % 1000)]).T
    Ng8_DDP  = N_g_8_theory(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

    # === 高效映射核心步骤 ===
    # 创建一个与原始 data 等长的数组，初始化为 NaN
    Ng8_full = np.full(N_total, np.nan)
    # 将算好的 Ng8 填入 DDP 星系所在的位置
    Ng8_full[mask_DDP] = Ng8_DDP
    # ========================
    for i in range(4):
        stat_bot_rsd = []
        stat_top_rsd = []
        ### Mr mask is related to the z_max 
        mask = np.logical_and(  np.logical_and(M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]  ),
                                np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]) )
        
        if j == 3:
            pi_max = 40.
        else :
            pi_max = 60.

        data_thr = data[mask]
        # === 高效提取对应星系的 Ng8 ===
        # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
        Ng8_thr  = Ng8_full[mask]
        
        # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
        # 如果你的物理逻辑保证了 thr 样本一定是 DDP 的子集，这一步不会剔除任何星系
        valid_idx = ~np.isnan(Ng8_thr)
        data_thr_valid = data_thr[valid_idx]
        Ng8_thr_valid  = Ng8_thr[valid_idx]
        # ==============================
        coords_gal_thr_rsd  = np.array([    np.array(data_thr_valid['x'] % 1000),
                                            np.array(data_thr_valid['y'] % 1000),    
                                            np.array(data_thr_valid['z'] % 1000)]).T
        
        NR = 10*len(data_thr['x'])
        R1_X = np.random.rand(NR) * 1000
        R1_Y = np.random.rand(NR) * 1000
        R1_Z = np.random.rand(NR) * 1000
        coords_rand_thr = np.array([R1_X, R1_Y, R1_Z]).T
        
        wprp_bottom20_rsd, wprp_top20_rsd = Ng8_percentile_wprp_theory_DDP(coords_gal_thr_rsd, coords_rand_thr, Ng8_thr_valid, pi_max, rp_bins, boxsize=1000)

        stat_bot_rsd.append(wprp_bottom20_rsd)
        stat_top_rsd.append(wprp_top20_rsd)

        output_dict1['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_bot_rsd for item in sublist])
        output_dict2['wprp_top20_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_top_rsd for item in sublist])

    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40', **output_dict1)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_top20_stat_pimax60_60_60_40', **output_dict2)

#### CCMD main box w/ rsd

In [ ]:
### CCMD main box with rsd
data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
N_total = len(data)

output_dict1 = collections.defaultdict(list)
output_dict2 = collections.defaultdict(list)
#########################################################################################################
for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
        ### DDP ###################################################################################
    mask_DDP = data['Mr'] < M_thr_arr_DDP[j]                                   
    data_DDP = data[mask_DDP]
    pos_DDP  = return_box_rsd_pos(data_DDP['x'], data_DDP['y'], data_DDP['z'], data_DDP['vz'],)
    Ng8_DDP  = N_g_8_theory(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

    # === 高效映射核心步骤 ===
    # 创建一个与原始 data 等长的数组，初始化为 NaN
    Ng8_full = np.full(N_total, np.nan)
    # 将算好的 Ng8 填入 DDP 星系所在的位置
    Ng8_full[mask_DDP] = Ng8_DDP
    # ========================
    for i in range(4):
        stat_bot_rsd = []
        stat_top_rsd = []
        ### Mr mask is related to the z_max 
        mask = np.logical_and(  np.logical_and(M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]  ),
                                np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]) )
        
        if j == 3:
            pi_max = 40.
        else :
            pi_max = 60.

        data_thr = data[mask]
        # === 高效提取对应星系的 Ng8 ===
        # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
        Ng8_thr  = Ng8_full[mask]
        
        # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
        # 如果你的物理逻辑保证了 thr 样本一定是 DDP 的子集，这一步不会剔除任何星系
        valid_idx = ~np.isnan(Ng8_thr)
        data_thr_valid = data_thr[valid_idx]
        Ng8_thr_valid  = Ng8_thr[valid_idx]
        # ==============================
        coords_gal_thr_rsd = return_box_rsd_pos(data_thr_valid['x'], data_thr_valid['y'], data_thr_valid['z'], data_thr_valid['vz'],)
        
        NR = 10*len(data_thr['x'])
        R1_X = np.random.rand(NR) * 1000
        R1_Y = np.random.rand(NR) * 1000
        R1_Z = np.random.rand(NR) * 1000
        coords_rand_thr = np.array([R1_X, R1_Y, R1_Z]).T
        
        wprp_bottom20_rsd, wprp_top20_rsd = Ng8_percentile_wprp_theory_DDP(coords_gal_thr_rsd, coords_rand_thr, Ng8_thr_valid, pi_max, rp_bins, boxsize=1000)

        stat_bot_rsd.append(wprp_bottom20_rsd)
        stat_top_rsd.append(wprp_top20_rsd)

        output_dict1['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_bot_rsd for item in sublist])
        output_dict2['wprp_top20_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_top_rsd for item in sublist])

    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_wprp_rsd_bottom20_pimax60_60_60_40', **output_dict1)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_wprp_rsd_top20_pimax60_60_60_40', **output_dict2)

#### CCMD 20Vars box, rsd

In [ ]:
### CCMD 20Vars box with rsd
# vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        # 'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]
vars = ['Var19', 'Var20', ]

for var in vars:
    data = fits.open("/share/hjxu/ccmdmock/Vars/" + var + "/CCMD_Mock_" + var + "_GalHaloInfo.fits")[1].data
    N_total = len(data)
    
    output_dict1 = collections.defaultdict(list)
    output_dict2 = collections.defaultdict(list)
    #########################################################################################################
    for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
        ### DDP ###################################################################################
        mask_DDP = data['Mr'] < M_thr_arr_DDP[j]                                   
        data_DDP = data[mask_DDP]
        pos_DDP  = return_box_rsd_pos(data_DDP['x'], data_DDP['y'], data_DDP['z'], data_DDP['vz'],)
        Ng8_DDP  = N_g_8_theory(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

        # === 高效映射核心步骤 ===
        # 创建一个与原始 data 等长的数组，初始化为 NaN
        Ng8_full = np.full(N_total, np.nan)
        # 将算好的 Ng8 填入 DDP 星系所在的位置
        Ng8_full[mask_DDP] = Ng8_DDP
        # ========================
        for i in range(4):
            stat_bot_rsd = []
            stat_top_rsd = []
            ### Mr mask is related to the z_max 
            mask = np.logical_and(  np.logical_and(M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]  ),
                                    np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]) )
            
            if j == 3:
                pi_max = 40.
            else :
                pi_max = 60.

            data_thr = data[mask]
            # === 高效提取对应星系的 Ng8 ===
            # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
            Ng8_thr  = Ng8_full[mask]
            
            # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
            # 如果你的物理逻辑保证了 thr 样本一定是 DDP 的子集，这一步不会剔除任何星系
            valid_idx = ~np.isnan(Ng8_thr)
            data_thr_valid = data_thr[valid_idx]
            Ng8_thr_valid  = Ng8_thr[valid_idx]
            # ==============================
            coords_gal_thr_rsd = return_box_rsd_pos(data_thr_valid['x'], data_thr_valid['y'], data_thr_valid['z'], data_thr_valid['vz'],)
            
            NR = 10*len(data_thr['x'])
            R1_X = np.random.rand(NR) * 1000
            R1_Y = np.random.rand(NR) * 1000
            R1_Z = np.random.rand(NR) * 1000
            coords_rand_thr = np.array([R1_X, R1_Y, R1_Z]).T
            
            wprp_bottom20_rsd, wprp_top20_rsd = Ng8_percentile_wprp_theory_DDP(coords_gal_thr_rsd, coords_rand_thr, Ng8_thr_valid, pi_max, rp_bins, boxsize=1000)

            stat_bot_rsd.append(wprp_bottom20_rsd)
            stat_top_rsd.append(wprp_top20_rsd)

            output_dict1['wprp_rsd_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_bot_rsd for item in sublist])
            output_dict2['wprp_rsd_top20_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat_top_rsd for item in sublist])

        np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_box_Mr_C_Ng8_wprp_rsd_bottom20_pimax60_60_60_40.npz', **output_dict1)
        np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_box_Mr_C_Ng8_wprp_rsd_top20_pimax60_60_60_40.npz', **output_dict2)

In [ ]:
vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]

output_dict_bottom_stat = collections.defaultdict(list)
output_dict_bottom_cov  = collections.defaultdict(list)
output_dict_top_stat = collections.defaultdict(list)
output_dict_top_cov  = collections.defaultdict(list)

for j in range(4): # Mr, z
    for i in range(4): # Color
        bottom_data  = []
        top_data     = []

        for var in vars:
            bottom_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_box_Mr_C_Ng8_wprp_rsd_bottom20_pimax60_60_60_40.npz')
            bottom_data.append(bottom_temp['wprp_rsd_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])
            top_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_' + var + '_box_Mr_C_Ng8_wprp_rsd_top20_pimax60_60_60_40.npz')
            top_data.append(top_temp['wprp_rsd_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

        bottom_data  = np.array(bottom_data)
        top_data     = np.array(top_data)

        output_dict_bottom_stat['bottom_stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.mean(bottom_data, axis=0)
        output_dict_bottom_cov[ 'bottom_cov_'  + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.cov( bottom_data, rowvar=False)
        output_dict_top_stat[   'top_stat_'    + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.mean(top_data, axis=0)
        output_dict_top_cov[    'top_cov_'     + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.cov( top_data, rowvar=False) 

    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_bottom_stat_mean.npz', **output_dict_bottom_stat)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_bottom_cov.npz'      , **output_dict_bottom_cov)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_top_stat_mean.npz'   , **output_dict_top_stat)
    np.savez('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_top_cov.npz'         , **output_dict_top_cov)

### Plot

In [ ]:
SDSS_Mr_C_bottom20        = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
SDSS_Mr_C_top20           = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')
SDSS_cov_Mr_C_bottom20    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_cov_pimax60_60_60_40.npz')
SDSS_cov_Mr_C_top20       = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_cov_pimax60_60_60_40.npz')
# SDSS_cov_Mr_C_bottom20_jk = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_bottom20_cov_jkknife_sample_pimax60_60_60_40.npz')
# SDSS_cov_Mr_C_top20_jk    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/SDSS_bright1_Mr_C_Ng8_top20_cov_jkknife_sample_pimax60_60_60_40.npz')

CCMD_32Quads_bottom20     = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_stat_mean.npz')
CCMD_32Quads_bottom20_cov = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_bottom20_pimax_60_60_60_40_cov.npz')
CCMD_32Quads_top20        = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_stat_mean.npz')
CCMD_32Quads_top20_cov    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_cov.npz')

In [ ]:
# CCMD_box_bottom20 = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
# CCMD_box_top20    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')

# CCMD_box_rsd_bot  = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_wprp_rsd_bottom20_pimax60_60_60_40.npz')
# CCMD_box_rsd_top  = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_box_Mr_C_Ng8_wprp_rsd_top20_pimax60_60_60_40.npz')

In [ ]:
# CCMD_20Vars_box_rsd_bottom_mean = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_bottom_stat_mean.npz')
# CCMD_20Vars_box_rsd_bottom_cov  = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_bottom_cov.npz'      )
# CCMD_20Vars_box_rsd_top_mean    = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_top_stat_mean.npz'   )
# CCMD_20Vars_box_rsd_top_cov     = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf_Ng8inDDP/CCMD_20Vars_box_rsd_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_top_cov.npz'         )

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
x_add = [1.5, -0.65, -0.65, 0.1]

for j in range(4):
    for i in range(4):
    ### CCMD 32Quads #############################################################################################################
        stat = CCMD_32Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
        # positive value
        mask = stat == stat
        ax[j,i].plot(rp_bins_x[mask],  stat[mask], 
                        color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
        ax[j,i].fill_between(rp_bins_x[mask],  stat[mask] - yerr[mask], stat[mask] + yerr[mask],
                                color = 'forestgreen', alpha = 0.3)
        
        # negative value
        ax[j,i].plot(rp_bins_x[mask], -stat[mask], 
                        color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
        ax[j,i].fill_between(rp_bins_x[mask], -stat[mask] - yerr[mask], -stat[mask] + yerr[mask],
                                color = 'forestgreen', alpha = 0.3, hatch = '///', facecolor = 'none')
    
    ### SDSS bright1 #############################################################################################################
        stat = SDSS_Mr_C_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(SDSS_cov_Mr_C_bottom20['wprp_bottom20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        # positive value
        mask = stat == stat
        ax[j,i].errorbar(rp_bins_x[mask],  stat[mask], yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 3, ms = 6, fmt = 'o', label = 'SDSS')
        # negative value
        ax[j,i].errorbar(rp_bins_x[mask], -stat[mask], yerr = yerr[mask], capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 2.5, ms = 8, fmt = 'v')
        
    # ### CCMD box ##################################################################################################################
    #     stat = CCMD_box_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
    #     # positive value
    #     mask = stat == stat
    #     ax[j,i].plot(rp_bins_x[mask],  stat[mask], color = 'black', alpha = 0.9, lw = 3, ls = '-' , label = 'CCMD box')
    #     # negative value
    #     mask = stat == stat
    #     ax[j,i].plot(rp_bins_x[mask], -stat[mask], color = 'black', alpha = 0.9, lw = 3, ls = '--')
    # ### CCMD box rsd ##############################################################################################################
    #     stat = CCMD_box_rsd_bot['wprp_rsd_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
    #     # positive value
    #     mask = stat == stat
    #     ax[j,i].plot(rp_bins_x[mask],  stat[mask], color = 'red', alpha = 0.9, lw = 3, ls = '-' , label = 'CCMD box rsd')
    #     # negative value
    #     mask = stat == stat
    #     ax[j,i].plot(rp_bins_x[mask], -stat[mask], color = 'red', alpha = 0.9, lw = 3, ls = '--')
    
#############################################################################################################   
        ax[j,i].loglog()
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        # ax[j,i].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
        # ax[j,i].text(2.7, 3e2  , Mr_thr_text[j], fontsize = 16)
        # ax[j,i].text(9  , 1.7e2, pimax_texts[j], fontsize = 14)
        # ax[j,i].set_xlim(0.15, 40)
        ax[j,i].set_ylim(0.5, 5000)
    
for j in range(4):
    # ax[j,0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
    # ax[j,0].text(5.2, 1.3e3, Co_thr_text[j,0], fontsize = 16, color = 'royalblue')
    # ax[j,3].text(3.8, 1.3e3, Co_thr_text[j,3], fontsize = 16, color = 'red')
    ax[j,3].text(50, 50, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=16,)
    ax[j,0].text(11, 1.2e3, 'bluest' , fontsize = 18, color = 'royalblue')
    ax[j,3].text(8 , 1.2e3, 'reddest', fontsize = 18, color = 'red')

# for i in range(4):
#     ax[3,i].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0,0].legend(fontsize = 16, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
# legend.get_texts()[1].set_color('black')
# legend.get_texts()[2].set_color('red')

fig.supxlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', y = 0.04)
fig.supylabel(r'$w_{\rm p}(r_{\rm p} | N_g^8)\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', x = 0.07)

# plt.suptitle(r'$N_g^8$ bottom $20\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
x_add = [1.5, -0.65, -0.65, 0.1]

for j in range(4):
    for i in range(4):
    ### CCMD 32Quads ##########################################################################################  
        # top_err16 = CCMD_32Quads_top20_err16['err16_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        # top_err84 = CCMD_32Quads_top20_err84['err84_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        ax[j,i].plot(rp_bins_x, CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]], 
                    color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
        # ax[j,i].fill_between(rp_bins_x, top_err16, top_err84, color = 'forestgreen', alpha = 0.3)

        yerr = np.sqrt(CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
        # yerr = CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        ax[j,i].fill_between(rp_bins_x, CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] - yerr,
                                        CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] + yerr, 
                                        color = 'forestgreen', alpha = 0.3)
        # for k in range(32):
        #     ax[3,3].plot(rp_bins_x, top_data[k,:], color = 'black', alpha = 0.9, lw = 3, ls = '-')

    ## SDSS ################################################################################################### 
        stat = SDSS_Mr_C_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        yerr = np.sqrt(SDSS_cov_Mr_C_top20['wprp_top20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        ax[j,i].errorbar(rp_bins_x, stat, yerr = yerr, color = 'orange', mfc = 'white', alpha = 0.9, 
                         capsize = 0, lw = 2.5, mew = 2.5, ms = 6, fmt = 'o', label = 'SDSS')
        # ax[j,i].scatter(rp_bins_x,  stat, color = 'white', edgecolor = 'orange', s = 64, linewidths = 3,
        #                 alpha = 0.9, marker = 'o', label = 'SDSS')

    # ### CCMD box ################################################################################################
    #     stat = CCMD_box_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
    #     ax[j,i].plot(rp_bins_x,  stat, color = 'black', lw = 3, ls = '-', alpha = 0.9, label = 'CCMD box')

    # ### CCMD box rsd ############################################################################################
    #     stat = CCMD_box_rsd_top['wprp_rsd_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
    #     ax[j,i].plot(rp_bins_x,  stat, color = 'red', lw = 3, ls = '-', alpha = 0.9, label = 'CCMD box rsd')

#############################################################################################################   
        ax[j,i].loglog()
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        # ax[j,i].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
        # ax[j,i].text(2.7, 3e2  , Mr_thr_text[j], fontsize = 16)
        # ax[j,i].text(9  , 1.7e2, pimax_texts[j], fontsize = 14)
        # ax[j,i].set_xlim(0.15, 40)
        ax[j,i].set_ylim(0.5, 5000)
    
for j in range(4):
    # ax[j,0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
    # ax[j,0].text(5.2, 1.3e3, Co_thr_text[j,0], fontsize = 16, color = 'royalblue')
    # ax[j,3].text(3.8, 1.3e3, Co_thr_text[j,3], fontsize = 16, color = 'red')
    ax[j,3].text(50, 50, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=16,)
    ax[j,0].text(11, 1.2e3, 'bluest' , fontsize = 18, color = 'royalblue')
    ax[j,3].text(8 , 1.2e3, 'reddest', fontsize = 18, color = 'red')

# for i in range(4):
#     ax[3,i].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0,0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
# legend.get_texts()[1].set_color('black')
# legend.get_texts()[2].set_color('red')

fig.supxlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', y = 0.04)
fig.supylabel(r'$w_{\rm p}(r_{\rm p} | N_g^8)\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', x = 0.07)

# plt.suptitle(r'$N_g^8$ top $20\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_top20.pdf', bbox_inches='tight')

#### DDP for box is meaningless
##### Since we use Mr<-20 for Mr[-22, -21], there must be a volume scale problem, which accertain that the calculation is correct.

In [ ]:
# fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True, dpi = 150)
# plt.subplots_adjust(hspace = 0, wspace = 0)

# pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
# x_add = [1.5, -0.65, -0.65, 0.1]

# for j in range(4):
#     for i in range(4):
# ### bottom 20% ##########################################################################################################################
#     ### CCMD 32Quads #############################################################################################################
#         stat = CCMD_32Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         cov  = CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         yerr = np.sqrt(cov.diagonal())

#         # positive value
#         mask = stat == stat
#         ax[j,i].plot(rp_bins_x[mask],  stat[mask], 
#                         color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 32 quads')
#         ax[j,i].fill_between(rp_bins_x[mask],  stat[mask] - yerr[mask], stat[mask] + yerr[mask],
#                                 color = 'forestgreen', alpha = 0.3) 
#         # ax[j,i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err
#         # ax[j,i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err   
#         # negative value
#         ax[j,i].plot(rp_bins_x[mask], -stat[mask], 
#                         color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
#         ax[j,i].fill_between(rp_bins_x[mask], -stat[mask] - yerr[mask], -stat[mask] + yerr[mask],
#                                 color = 'forestgreen', alpha = 0.3, hatch = '///', facecolor = 'none')
#         # ax[j,i].fill_between(rp_bins_x, -(stat - yerr_total), -(stat - yerr), color = 'royalblue', 
#         #                     hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
#         # ax[j,i].fill_between(rp_bins_x, -(stat + yerr), -(stat + yerr_total), color = 'royalblue', 
#         #                     hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
    
#     ### CCMD box ##################################################################################################################
#         stat = CCMD_box_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         # positive value
#         mask = stat == stat
#         ax[j,i].plot(rp_bins_x[mask],  stat[mask], color = 'k', alpha = 0.9, lw = 3, ls = '-' , label = 'CCMD box w/o rsd')
#         # negative value
#         mask = stat == stat
#         ax[j,i].plot(rp_bins_x[mask], -stat[mask], color = 'k', alpha = 0.9, lw = 3, ls = '--')
#     ### CCMD box rsd ##############################################################################################################
#         stat = CCMD_box_rsd_bot['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         # positive value
#         mask = stat == stat
#         ax[j,i].plot(rp_bins_x[mask],  stat[mask], color = 'red', alpha = 0.9, lw = 3, ls = '-',  label = 'CCMD box w/ rsd')
#         # negative value
#         mask = stat == stat
#         ax[j,i].plot(rp_bins_x[mask], -stat[mask], color = 'red', alpha = 0.9, lw = 3, ls = '--')
#     ### CCMD 20Vars box, rsd #############################################################################################################
#         stat = CCMD_20Vars_box_rsd_bottom_mean['bottom_stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         yerr = np.sqrt(CCMD_20Vars_box_rsd_bottom_cov[ 'bottom_cov_'  + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
#         # positive value
#         mask = stat == stat
#         ax[j,i].plot(rp_bins_x[mask],  stat[mask], 
#                         color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
#         ax[j,i].fill_between(rp_bins_x[mask],  stat[mask] - yerr[mask], stat[mask] + yerr[mask],
#                                 color = 'royalblue', alpha = 0.3)     
#         # negative value
#         ax[j,i].plot(rp_bins_x[mask], -stat[mask], 
#                         color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
#         ax[j,i].fill_between(rp_bins_x[mask], -stat[mask] - yerr[mask], -stat[mask] + yerr[mask],
#                                 color = 'royalblue', alpha = 0.3, hatch = '///', facecolor = 'none')
# ### top 20% ############################################################################################################################
#     ### CCMD 32Quads ##########################################################################################
#         stat = CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         cov  = CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         yerr = np.sqrt(cov.diagonal())
#         ax[j,i].plot(rp_bins_x, stat, color = 'forestgreen', alpha = 0.9, lw = 2, ls = '-')#, label = 'CCMD 32 quads')
#         ax[j,i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.1) 
#         # cov_20Vars = CCMD_20Vars_box_rsd_top_cov[ 'top_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         # yerr_total = np.sqrt((cov + cov_20Vars).diagonal())    
#         # ax[j,i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
#         # ax[j,i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err                                  
#     ### CCMD box ################################################################################################
#         stat = CCMD_box_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         ax[j,i].plot(rp_bins_x,  stat, color = 'k', lw = 2, ls = '-', alpha = 0.9,)# label = 'CCMD box w/o rsd')
#     ### CCMD box rsd ############################################################################################
#         stat = CCMD_box_rsd_top['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         ax[j,i].plot(rp_bins_x,  stat, color = 'red', lw = 2, ls = '-', alpha = 0.9,)# label = 'CCMD box w/ rsd')
#     ### CCMD 20Vars box, rsd ##########################################################################################  
#         stat = CCMD_20Vars_box_rsd_top_mean['top_stat_'+ M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         yerr = np.sqrt(CCMD_20Vars_box_rsd_top_cov[ 'top_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
#         ax[j,i].plot(rp_bins_x, stat, color = 'royalblue', alpha = 0.9, lw = 2, ls = '-')#, label = 'CCMD 32 quads')
#         ax[j,i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'royalblue', alpha = 0.2) 
# #############################################################################################################   
#         ax[j,i].loglog()
#         ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
#         ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
#         ax[j,i].tick_params(axis='both', which='major', width=2, length=8)
#         ax[j,i].tick_params(axis='x', which='minor', width=2 , length=4, labelsize = 24)
#         ax[j,i].tick_params(axis='y', which='minor', length=0, labelsize = 24)
#         ax[j,i].set_ylim(0.5, 5000)
#         for spine in ax[j,i].spines.values():
#             # 对于主图 - 分别调整每个边框
#             ax[j,i].spines['top'].set_linewidth(2)
#             ax[j,i].spines['bottom'].set_linewidth(2) 
#             ax[j,i].spines['left'].set_linewidth(2)
#             ax[j,i].spines['right'].set_linewidth(2)
    
# for j in range(4):
#     # ax[j,0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
#     # ax[j,3].text(55 , 50  , Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=17,)
#     ax[j,3].text(1.4, 900 , Mr_thr_text[j], fontsize=20,)
#     ax[0,0].text(1.1, 120 , 'top 20%', fontsize = 16, color = 'dimgray')
#     ax[0,0].text(1.1, 1.0 , 'bot 20%', fontsize = 16, color = 'dimgray')

# # for i in range(4):
# #     ax[3,i].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

# ax[0,0].set_title(r'$blue$'   , fontsize = 24, color = 'royalblue')
# ax[0,1].set_title(r'$green$'  , fontsize = 24, color = 'forestgreen')
# ax[0,2].set_title(r'$red$'    , fontsize = 24, color = 'red')
# ax[0,3].set_title(r'$redder$' , fontsize = 24, color = 'red')

# legend = ax[3,0].legend(fontsize = 13, loc = 'upper right',frameon = False) 
# legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[2].set_color('red')
# legend.get_texts()[3].set_color('royalblue')

# fig.supxlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$', fontsize=26, fontweight='bold', y = 0.03)
# fig.supylabel(r'$w_{\rm p}(r_{\rm p} | \tilde{N}_g^8)$', fontsize=26, fontweight='bold', x = 0.065)

# # plt.suptitle(r'$N_g^8$ bottom $20\%$', y =0.95)
# # plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
x_add = [1.5, -0.65, -0.65, 0.1]

for j in range(4):
    for i in range(4):
    ### CCMD 32Quads #############################################################################################################
        stat = CCMD_32Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        # yerr = np.sqrt(CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
        yerr = CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        ax[j,i].plot(np.abs(yerr/stat), color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
        ax[j,i].axhline(y=1, color='lightgray', linestyle='--')
    ### SDSS bright1 #############################################################################################################
        stat = SDSS_Mr_C_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
            
        yerr = np.sqrt(SDSS_cov_Mr_C_bottom20['wprp_bottom20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        
        ax[j,i].plot(np.abs(yerr/stat),color = 'orange', ls = '--', label = 'SDSS')
#############################################################################################################   
        ax[j,i].loglog()
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        ax[j,i].text(3.7+x_add[i],1.3e2, Co_thr_text[j,i], fontsize = 16)
        ax[j,i].text(2.7, 3e1  , Mr_thr_text[j], fontsize = 16)
        # ax[j,i].text(9  , 1.7e2, pimax_texts[j], fontsize = 14)
        ax[j,i].set_xlim(0.15, 40)
        ax[j,i].set_ylim(0.1, 500)
    
for j in range(4):
    ax[j,0].set_ylabel('|yerr/stat|',fontsize = 16,)
    ax[j,0].text(5.2, 1.3e2, Co_thr_text[j,0], fontsize = 16, color = 'royalblue')
    ax[j,3].text(3.8, 1.3e2, Co_thr_text[j,3], fontsize = 16, color = 'red')

for i in range(4):
    ax[3,i].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0,0].legend(fontsize = 16, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')

plt.suptitle(r'$N_g^8$ bottom $20\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
x_add = [1.5, -0.65, -0.65, 0.1]

for j in range(4):
    for i in range(4):
    ### CCMD 32Quads
        stat = CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
        
        ax[j,i].plot(np.abs(yerr/stat), color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
        ax[j,i].axhline(y=1, color='lightgray', linestyle='--')
    ### SDSS
        stat = SDSS_Mr_C_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        yerr = np.sqrt(SDSS_cov_Mr_C_top20['wprp_top20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        
        ax[j,i].plot(np.abs(yerr/stat),color = 'orange', ls = '--', label = 'SDSS')

#############################################################################################################   
        ax[j,i].loglog()
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        ax[j,i].text(3.7+x_add[i],1.3e2, Co_thr_text[j,i], fontsize = 16)
        ax[j,i].text(2.7, 3e1  , Mr_thr_text[j], fontsize = 16)
        # ax[j,i].text(9  , 1.7e2, pimax_texts[j], fontsize = 14)
        ax[j,i].set_xlim(0.15, 40)
        ax[j,i].set_ylim(0.01, 500)
    
for j in range(4):
    ax[j,0].set_ylabel('|yerr/stat|',fontsize = 16,)
    ax[j,0].text(5.2, 1.3e2, Co_thr_text[j,0], fontsize = 16, color = 'royalblue')
    ax[j,3].text(3.8, 1.3e2, Co_thr_text[j,3], fontsize = 16, color = 'red')

for i in range(4):
    ax[3,i].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0,0].legend(fontsize = 16, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')

plt.suptitle(r'$N_g^8$ top $20\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_top20.pdf', bbox_inches='tight')

In [ ]:
# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# output_dict_bottom_stat = collections.defaultdict(list)
# output_dict_top_stat    = collections.defaultdict(list)
# output_dict_bottom_cov  = collections.defaultdict(list)
# output_dict_top_cov     = collections.defaultdict(list)

# for j in [3]: # Mr, z
#     for i in [3]: # Color
#         bottom_data  = []
#         top_data     = []
#         for var in vars:
#             for quad in quads:
#                 bottom_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_stat_pimax60_60_60_40.npz')
#                 bottom_data.append(bottom_temp['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

#                 top_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_2pcf/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')
#                 top_data.append(top_temp['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

#         bottom_data  = np.array(bottom_data)
#         top_data     = np.array(top_data)

In [ ]:
np.shape(top_data)

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
x_add = [1.5, -0.65, -0.65, 0.1]

for j in range(4):
    for i in range(4):
### bottom 20% #########################################################################################################################
    ### CCMD 32Quads #############################################################################################################
        stat = CCMD_32Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        cov  = CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(cov.diagonal())
        cov_20Vars = CCMD_20Vars_box_rsd_bottom_cov['bottom_cov_'  + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr_total = np.sqrt((cov + cov_20Vars).diagonal())

        # positive value
        # mask = stat == stat
        ax[j,i].plot(rp_bins_x,  stat,  color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD')
        ax[j,i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3)
        ax[j,i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err
        ax[j,i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err  
        # negative value
        ax[j,i].plot(rp_bins_x, -stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
        ax[j,i].fill_between(rp_bins_x, -stat - yerr, -stat + yerr, color = 'forestgreen', alpha = 0.3, hatch = '///', facecolor = 'none')
        ax[j,i].fill_between(rp_bins_x, -(stat - yerr_total), -(stat - yerr), color = 'royalblue', 
                            hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
        ax[j,i].fill_between(rp_bins_x, -(stat + yerr), -(stat + yerr_total), color = 'royalblue', 
                            hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
    ### SDSS bright1 #############################################################################################################
        stat = SDSS_Mr_C_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(SDSS_cov_Mr_C_bottom20['wprp_bottom20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        # positive value
        mask = stat == stat
        ax[j,i].errorbar(rp_bins_x,  stat, yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 3, ms = 6, fmt = 'o')#, label = 'SDSS')
        # negative value
        ax[j,i].errorbar(rp_bins_x, -stat, yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 2.5, ms = 8, fmt = 'v')
### top 20% ############################################################################################################################
    ### CCMD 32Quads ########################################################################################## 
        stat = CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        cov  = CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(cov.diagonal())
        ax[j,i].plot(rp_bins_x, stat, color = 'forestgreen', alpha = 0.7, lw = 2, ls = '-', label = 'CCMD')
        ax[j,i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.1)
        cov_20Vars = CCMD_20Vars_box_rsd_top_cov[ 'top_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr_total = np.sqrt((cov + cov_20Vars).diagonal())    
        ax[j,i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
        ax[j,i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
    ## SDSS ################################################################################################### 
        stat = SDSS_Mr_C_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(SDSS_cov_Mr_C_top20['wprp_top20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        ax[j,i].errorbar(rp_bins_x, stat, yerr = yerr, color = 'orange', mfc = 'white', alpha = 0.7, 
                         capsize = 0, lw = 2, mew = 2, ms = 5, fmt = 'o', label = 'SDSS')

#############################################################################################################   
        ax[j,i].loglog()
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis='both', which='major', width=1.5, length=6)
        ax[j,i].tick_params(axis='both', which='minor', width=1.0, length=3)
        # ax[j,i].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
        # ax[j,i].text(2.7, 3e2  , Mr_thr_text[j], fontsize = 16)
        # ax[j,i].text(9  , 1.7e2, pimax_texts[j], fontsize = 14)
        # ax[j,i].set_xlim(0.15, 40)
        ax[j,i].set_ylim(0.2, 5000)
    
for j in range(4):
    # ax[j,0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
    # ax[j,0].text(5.2, 1.3e3, Co_thr_text[j,0], fontsize = 16, color = 'royalblue')
    # ax[j,3].text(3.8, 1.3e3, Co_thr_text[j,3], fontsize = 16, color = 'red')
    # ax[j,3].text(55, 50, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=17,)
    ax[j,0].text(0.2, 1100, Mr_thr_text[j], fontsize=20,)
    # ax[j,0].text(11, 1.2e3, 'bluest' , fontsize = 18, color = 'royalblue')
    # ax[j,3].text(8 , 1.2e3, 'reddest', fontsize = 18, color = 'red')
    # ax[1,0].text(1.5, 120 , 'overdense' , fontsize = 15, color = 'gray', alpha = 1)
    # ax[1,0].text(1.5, 0.5 , 'underdense', fontsize = 15, color = 'gray', alpha = 1)
    ax[1,0].text(1.8, 100 , 'top 20%', fontsize = 18, color = 'dimgray', alpha = 1)
    ax[1,0].text(1.8, 0.5 , 'bot 20%', fontsize = 18, color = 'dimgray', alpha = 1)

# for i in range(4):
#     ax[3,i].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0,1].legend(fontsize = 20, loc = 'upper right',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
# legend.get_texts()[1].set_color('black')
# legend.get_texts()[2].set_color('red')

ax[0,0].set_title('blue', fontsize = 22, color = 'royalblue')
ax[0,1].set_title('green', fontsize = 22, color = 'forestgreen')
ax[0,2].set_title('red'    , fontsize = 22, color = 'red')
ax[0,3].set_title('more red' , fontsize = 22, color = 'red')

fig.supxlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$', fontsize=24, fontweight='bold', y = 0.04)
fig.supylabel(r'$w_{\rm p}(r_{\rm p} | N_g^8)$', fontsize=24, fontweight='bold', x = 0.07)
# plt.tight_layout()
# plt.suptitle(r'$N_g^8$ bottom $20\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(25, 15), dpi = 150)

# 创建主GridSpec：4行（每行包含原图+比值图），4列
gs_main = gridspec.GridSpec(4, 4, figure=fig, height_ratios=[5, 5, 5, 5], hspace=0.05, wspace=0)

ax_main = [[None for _ in range(4)] for _ in range(4)]  # 原图
ax_ratio = [[None for _ in range(4)] for _ in range(4)]  # 比值图

percent = 100

for j in range(4):
    for i in range(4):
        # 为每个单元格创建嵌套的GridSpec (2行1列，高度比例4:1)
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[j, i], 
                                                  height_ratios=[10, 5], hspace=0)
        ########################################################################################################################################
        # 原图（上部分）
        ax_main[j][i] = fig.add_subplot(gs_cell[0])
### bottom 20% #########################################################################################################################
        ### CCMD 32Quads #############################################################################################################
        stat = CCMD_32Quads_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        cov  = CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(cov.diagonal())
        # cov_20Vars = CCMD_20Vars_box_rsd_bottom_cov['bottom_cov_'  + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        # yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
        yerr_total = np.sqrt((cov).diagonal())
        # positive value
        # mask = stat == stat
        ax_main[j][i].plot(rp_bins_x,  stat,  color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD')
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3)
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err
        ax_main[j][i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err  
        # negative value
        ax_main[j][i].plot(rp_bins_x, -stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
        ax_main[j][i].fill_between(rp_bins_x, -stat - yerr, -stat + yerr, color = 'forestgreen', alpha = 0.3, hatch = '///', facecolor = 'none')
        ax_main[j][i].fill_between(rp_bins_x, -(stat - yerr_total), -(stat - yerr), color = 'royalblue', 
                            hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
        ax_main[j][i].fill_between(rp_bins_x, -(stat + yerr), -(stat + yerr_total), color = 'royalblue', 
                            hatch = '///', facecolor = 'none', alpha = 0.3,) # add CCMD 20Vars box err
        ### SDSS bright1 #############################################################################################################
        stat = SDSS_Mr_C_bottom20['wprp_bottom20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(SDSS_cov_Mr_C_bottom20['wprp_bottom20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        # positive value
        mask = stat == stat
        ax_main[j][i].errorbar(rp_bins_x,  stat, yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 3, ms = 6, fmt = 'o')#, label = 'SDSS')
        # negative value
        ax_main[j][i].errorbar(rp_bins_x, -stat, yerr = yerr, capsize = 0, lw = 2.5,
                         color = 'orange', mfc = 'white', alpha = 0.9, mew = 2.5, ms = 8, fmt = 'v')
### top 20% ############################################################################################################################
        ### CCMD 32Quads ########################################################################################## 
        stat = CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        cov  = CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        yerr = np.sqrt(cov.diagonal())
        ax_main[j][i].plot(rp_bins_x, stat, color = 'forestgreen', alpha = 0.7, lw = 2, ls = '-', label = 'CCMD')
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.1)
        # cov_20Vars = CCMD_20Vars_box_rsd_top_cov[ 'top_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        # yerr_total = np.sqrt((cov + cov_20Vars).diagonal())    
        yerr_total = np.sqrt((cov).diagonal())    
        ax_main[j][i].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
        ax_main[j][i].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.1,) # add CCMD 20Vars box err
        ## SDSS ################################################################################################### 
        stat_SDSS = SDSS_Mr_C_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        yerr_SDSS = np.sqrt(SDSS_cov_Mr_C_top20['wprp_top20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())[1:13]
        ax_main[j][i].errorbar(rp_bins_x, stat_SDSS, yerr = yerr_SDSS, color = 'orange', mfc = 'white', alpha = 0.7, 
                         capsize = 0, lw = 2, mew = 2, ms = 5, fmt = 'o', label = 'SDSS')

        ########################################################################################################################################
        # 比值图（下部分）
        ax_ratio[j][i] = fig.add_subplot(gs_cell[1])
        # 在比值图中绘图（替换为您的比值数据）
        # ratio_data = y / y.max()  # 示例数据
        # ax_ratio[j][i].plot(x, ratio_data, color='red', linewidth=1)
        ### CCMD
        ax_ratio[j][i].plot(rp_bins_x, (stat/stat-1)*percent, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-',)# label = 'CCMD')
        ax_ratio[j][i].fill_between(rp_bins_x, ((stat - yerr)/stat-1)*percent, ((stat + yerr)/stat-1)*percent, color = 'forestgreen', alpha = 0.3,)
        ax_ratio[j][i].fill_between(rp_bins_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) # add CCMD 20Vars box err
        ax_ratio[j][i].fill_between(rp_bins_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) # add CCMD 20Vars box err
        ### SDSS
        ax_ratio[j][i].errorbar(rp_bins_x, (stat_SDSS/stat-1)*percent, yerr = (yerr_SDSS/stat)*percent, 
                   color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                   marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white',)#  label = 'SDSS')
        ### ratio = 1
        ax_ratio[j][i].plot([-5, 60], [0, 0], color = 'gray', alpha = 0.6)
        ##########################################################################        
        # 调整主图四个边框的粗细
        for spine in ax_main[j][i].spines.values():
                # 对于主图 - 分别调整每个边框
                ax_main[j][i].spines['top'].set_linewidth(2)
                ax_main[j][i].spines['bottom'].set_linewidth(1) 
                ax_main[j][i].spines['left'].set_linewidth(2)
                ax_main[j][i].spines['right'].set_linewidth(2)
        for spine in ax_ratio[j][i].spines.values():
                # 对于比值图 - 分别调整每个边框
                ax_ratio[j][i].spines['top'].set_linewidth(1)
                ax_ratio[j][i].spines['bottom'].set_linewidth(2)
                ax_ratio[j][i].spines['left'].set_linewidth(2)
                ax_ratio[j][i].spines['right'].set_linewidth(2)

        ax_ratio[j][i].set_xscale('log')
        if j < 2:
            # ax_ratio[j][i].set_yticks([-30, 30], ['-30%', '30%'])
            # ax_ratio[j][i].set_ylim(-50, 50)
            ax_ratio[j][i].set_yticks([-50, 50], ['-50%', '50%'])
            ax_ratio[j][i].set_ylim(-90, 90)
        else:
            ax_ratio[j][i].set_yticks([-70, 70], ['-70%', '70%'])
            ax_ratio[j][i].set_ylim(-130, 130)
        ax_ratio[j][i].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_ratio[j][i].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_ratio[j][i].tick_params(axis='both', which='major', width=2, length=8, labelsize=24,)
        ax_ratio[j][i].tick_params(axis='both', which='minor', width=2, length=5, labelsize=24,)
        # ax_ratio[j][i].xaxis.set_minor_locator(plt.MultipleLocator(0.5))  # x轴子刻度
        # ax_ratio[j][i].yaxis.set_minor_locator(plt.MultipleLocator(0.5))  # y轴子刻度
        ax_ratio[j][i].set_xlim(0.15, 40)
        # ax_ratio[j][i].set_xlim(-0.9, 1.7)

        ax_main[j][i].loglog()
        # # ax_main[j][i].set_xscale('log')
        # ax_main[j][i].set_yticks([1, 3, 5], ['1', '3', '5'])
        ax_main[j][i].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_main[j][i].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_main[j][i].tick_params(axis='both', which='major', width=2, length=8, labelsize=24,)
        ax_main[j][i].tick_params(axis='both', which='minor', width=2, length=5, labelsize=24,)
        # ax_main[j][i].xaxis.set_minor_locator(plt.MultipleLocator(0.5))  # x轴子刻度
        # ax_main[j][i].yaxis.set_minor_locator(plt.MultipleLocator(1))  # y轴子刻度
        ax_main[j][i].set_xlim(0.15, 40)
        ax_main[j][i].set_ylim(0.5, 5000)
        # ax_main[j][i].set_xlim(-0.9, 1.7)
        # ax_main[j][i].set_ylim(0.1, 5.5)
            
for j in range(3):
    for i in [1, 2, 3]:
        ax_ratio[j][i].tick_params(axis='both', which='major', labelsize=24, labelbottom=False, labelleft=False)
        ax_ratio[j][i].tick_params(axis='both', which='minor', labelsize=24, labelbottom=False, labelleft=False)
        ax_ratio[j][0].tick_params(axis='both', which='major', labelsize=24, labelbottom=False)
        ax_ratio[j][0].tick_params(axis='both', which='minor', labelsize=24, labelbottom=False)
        ax_ratio[3][i].tick_params(axis='both', which='major', labelsize=24, labelleft=False)
        ax_ratio[3][i].tick_params(axis='both', which='minor', labelsize=24, labelleft=False)
     
for j in range(4):
    ax_main[j][0].text(0.2, 500, Mr_thr_text[j], fontsize=22, fontweight='bold',)  
    ax_main[0][0].text(1.6, 100, 'top 20%', rotation=0, fontsize = 15, color = 'dimgray', alpha = 1)
    ax_main[0][0].text(1.6, 1  , 'bot 20%', rotation=0, fontsize = 15, color = 'dimgray', alpha = 1)
#     ax_main[j][3].text(55, 15, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=24, fontweight='bold',)
#     ax_main[j][3].text(1.9, 1.5, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=24, fontweight='bold',)
    for i in [1, 2, 3]:
        ax_main[j][i].tick_params(axis='both', which='major', labelsize=24, labelbottom=False, labelleft=False)
        ax_main[j][i].tick_params(axis='both', which='minor', labelsize=24, labelbottom=False, labelleft=False)
    for i in range(4):
        ax_ratio[j][i].tick_params(axis = 'y', direction = 'in', which = u'both', labelsize = 18)
        ax_ratio[0][0].text(1, -50, 'Diff. (top 20%)', fontsize = 18, fontweight='normal', color = 'k')
         
legend = ax_main[3][0].legend(fontsize = 20, loc = 'upper right',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('royalblue')
legend.get_texts()[1].set_color('orange')

ax_main[0][0].set_title(r'$blue$'   , fontsize = 30, fontweight='medium', color = 'royalblue')
ax_main[0][1].set_title(r'$green$'  , fontsize = 30, fontweight='medium', color = 'forestgreen')
ax_main[0][2].set_title(r'$red$'    , fontsize = 30, fontweight='medium', color = 'red')
ax_main[0][3].set_title(r'$redder$' , fontsize = 30, fontweight='medium', color = 'red')

fig.supxlabel(r'$r_{ p}~[h^{-1}{\rm Mpc}]$'  , fontsize=30, fontweight='bold', y = 0.04)
fig.supylabel(r'$w_{ p}(r_{ p} | \tilde{N}_g^8)$', fontsize=30, fontweight='bold', x = 0.07)

plt.tight_layout()
plt.show()

In [ ]:
CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

In [ ]:
def cov_to_corr(input_stat, cov):
    variances = np.diag(cov) 
    zero_element = variances == 0
    zero_elements_indices = np.where(variances == 0)[0]
    input_stat = input_stat[~zero_element]
    # bestfit = bestfit[~zero_element]
    ### Remove the ith row and column
    cov = np.delete(np.delete(cov, zero_elements_indices, axis=0), zero_elements_indices, axis=1)
    ### Normalize the covariance matrix
    stdv = np.sqrt(np.diag(cov))  # Square root of the diagonal elements
    outer_diag = np.outer(stdv, stdv)   # Outer product of the diagonal elements
    corr = cov / outer_diag
    return input_stat, corr, stdv

In [ ]:
CCMD_32Quads_top20        = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_stat_mean.npz')
CCMD_32Quads_top20_cov    = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_pimax_60_60_60_40_cov.npz')

i = 3
j = 3
input_stat, corr_matrix, stdv = cov_to_corr(CCMD_32Quads_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]], 
                                            CCMD_32Quads_top20_cov['wprp_top20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])
# Plot Correlation Coviriance Matrix
plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Correlation')
plt.title('CCMD Ng8 top, ' + Mr_thr_text[j] + ', ' + Co_thr_text[j,i], fontsize = 14)
plt.show()

In [ ]:
SDSS_Mr_C_top20           = np.load('../CCMD_mag_bin/wp_Mr_C_Ng8/SDSS_bright1_Mr_C_Ng8_top20_stat_pimax60_60_60_40.npz')
SDSS_cov_Mr_C_top20       = np.load('../CCMD_mag_bin/wp_Mr_C_Ng8/SDSS_bright1_Mr_C_Ng8_top20_cov_pimax60_60_60_40.npz')

i = 3
j = 3
input_stat, corr_matrix, stdv = cov_to_corr(SDSS_Mr_C_top20['wprp_top20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]], 
                                            SDSS_cov_Mr_C_top20['wprp_top20_cov' + M_thr_list[j] + '_C_' + C_thr_list[j,i]][1:13, 1:13])
# Plot Correlation Coviriance Matrix
plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Correlation')
plt.title('SDSS Ng8 top, ' + Mr_thr_text[j] + ', ' + Co_thr_text[j,i], fontsize = 14)
plt.show()

#### Test Ng8 in hist, quad in DDP

In [ ]:
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']
quad = 'Quad0'
var  = 'Observer_1'
data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
N_total = len(data)

### >>> 新增：初始化一个空字典用于存储数据
output_data = {}

#########################################################################################################
for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
    ### DDP ###################################################################################
    mask_DDP = np.logical_and(  data['Mr'] < M_thr_arr_DDP[j],
                                        np.logical_and(z_min <= data['z'], data['z'] < z_max_arr_DDP[j] ) )
    data_DDP = data[mask_DDP]
    pos_DDP  = np.array([   np.array(data_DDP['ra']),
                            np.array(data_DDP['dec']),    
                            np.array(data_DDP['z'])]).T
    Ng8_DDP  = N_g_8(   pos_DDP[:, 0], pos_DDP[:, 1], pos_DDP[:, 2] )

    # === 高效映射核心步骤 ===
    # 创建一个与原始 data 等长的数组，初始化为 NaN
    Ng8_full = np.full(N_total, np.nan)
    # 将算好的 Ng8 填入 DDP 星系所在的位置
    Ng8_full[mask_DDP] = Ng8_DDP
    # ========================

    for i in range(4):
        ### Mr mask is related to the z_max 
        mask = np.logical_and(  np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))

        data_thr  = data[mask]
        # 直接通过同样的 mask 提取出当前 bin 中星系的 Ng8
        Ng8_thr   = Ng8_full[mask]
        
        # 安全检查：如果 data_thr 中包含了不在 data_DDP 中的星系，其 Ng8 会是 NaN
        valid_idx = ~np.isnan(Ng8_thr)
        data_thr_valid  = data_thr[valid_idx]
        Ng8_thr_valid   = Ng8_thr[valid_idx]
        # ==============================


        ### >>> 新增：将计算结果以 "变量名_Mr_Color" 为 key 存入字典
        key_suffix = M_thr_list[j] + '_C_' + C_thr_list[j,i]
        output_data['Ng8_thr_valid_' + key_suffix] = Ng8_thr_valid

       
### >>> 新增：循环结束后，将字典打包存为本地 npz 文件
np.savez('./CCMD_' + var + '_' + quad + '_Ng8_data_DDP_test.npz', **output_data)
print('Data successfully saved to ./CCMD_' + var + '_' + quad + '_Ng8_data_DDP_test.npz')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 读取保存好的 npz 数据
# 请确保文件名与你上一步保存的一致
saved_data = np.load('./CCMD_' + var + '_' + quad + '_Ng8_data_DDP_test.npz')

# 2. 创建 4x4 画布
fig, ax = plt.subplots(4, 4, figsize=(20, 12), sharey=True, sharex=True)
plt.subplots_adjust(hspace=0, wspace=0)

for j in range(4): # 对应 Mr bins
    for i in range(4): # 对应 Color bins
        
        # 构造读取字典的 key
        key_suffix = M_thr_list[j] + '_C_' + C_thr_list[j,i]
        
        # 提取当前 panel 的 Ng8 数组和 bottom 20% 的 fsat
        Ng8_arr = saved_data['Ng8_thr_valid_' + key_suffix]

        # ### only bottom 20% =========================================
        Ng8_arr_bot20 = np.sort(Ng8_arr)[:int(len(Ng8_arr) * 0.2)]

        # === 核心计算：均值和中位数 ===
        mean_val = np.mean(Ng8_arr_bot20)
        median_val = np.median(Ng8_arr_bot20)

        # 绘制直方图 (加上边缘线会让柱子更清晰)
        n_bins = int(np.ceil(Ng8_arr_bot20.max() - Ng8_arr_bot20.min()))
        ax[j,i].hist(Ng8_arr_bot20, bins=max(1, n_bins), density=True, 
                     color='lightgray', edgecolor='gray', alpha=0.7)
        #===============================================================
        
        # === 新增：绘制均值和中位数竖线 ===
        ax[j,i].axvline(mean_val, color='red', linestyle='--', linewidth=2, 
                        label=f'Mean: {mean_val:.1f}')
        ax[j,i].axvline(median_val, color='blue', linestyle=':', linewidth=2, 
                        label=f'Median: {median_val:.1f}')

        # === 标注文本信息 ===
        # 1. 标明这是哪个光度-颜色区间 (使用 transAxes 定位，0.5代表居中)
        ax[j,i].text(0.5, 0.8, f'{Mr_thr_text[j]}, {Co_thr_text[j,i]}', 
                     transform=ax[j,i].transAxes, fontsize=12, ha='center', weight='bold')
        
        # 2. 标明 Ngal in fine bins (放在左侧居中位置)
        
        ax[j,i].text(0.4, 0.2, r'$N_{gal}^{fine-bin}$ = ' + f'{len(Ng8_arr)}', 
                     transform=ax[j,i].transAxes, fontsize=12, ha='left')
        
        # 2. 添加图例，显示 Mean 和 Median 的数值 (放在右侧)
        ax[j,i].legend(loc='center right', fontsize=10, frameon=False)

        # 设置坐标轴标签和范围
        ax[3,i].set_xlabel(r'$N_g^8$', fontsize=14)
        ax[j,0].set_ylabel(r'$P(N_g^8)$', fontsize=14)
        
        # ax[j,i].set_xlim(0, 225)
        ax[j,i].set_xlim(0, 75)
        # ax[j,i].set_ylim(0, 0.05)

plt.suptitle(r'$N_g^8$ Distribution (DDP defined) with Mean & Median, bottom 20%, ' + var + '_' + quad, y=0.92, fontsize=16, weight='bold')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 读取保存好的 npz 数据
# 请确保文件名与你上一步保存的一致
saved_data = np.load('./CCMD_' + var + '_' + quad + '_Ng8_data_DDP_test.npz')

# 2. 创建 4x4 画布
fig, ax = plt.subplots(4, 4, figsize=(20, 12), sharey=True, sharex=True)
plt.subplots_adjust(hspace=0, wspace=0)

for j in range(4): # 对应 Mr bins
    for i in range(4): # 对应 Color bins
        
        # 构造读取字典的 key
        key_suffix = M_thr_list[j] + '_C_' + C_thr_list[j,i]
        
        # 提取当前 panel 的 Ng8 数组和 top 20% 的 fsat
        Ng8_arr = saved_data['Ng8_thr_valid_' + key_suffix]
        
        # only top 20% =========================================
        Ng8_arr_top20 = np.sort(Ng8_arr)[-int(len(Ng8_arr) * 0.2):]

        # === 核心计算：均值和中位数 ===
        mean_val = np.mean(Ng8_arr_top20)
        median_val = np.median(Ng8_arr_top20)

        # 绘制直方图 (加上边缘线会让柱子更清晰)
        n_bins = int(np.ceil(Ng8_arr_top20.max() - Ng8_arr_top20.min()))
        ax[j,i].hist(Ng8_arr_top20, bins=max(1, n_bins), density=True, 
                     color='lightgray', edgecolor='gray', alpha=0.7)
        # ======================================================
        
        # === 新增：绘制均值和中位数竖线 ===
        ax[j,i].axvline(mean_val, color='red', linestyle='--', linewidth=2, 
                        label=f'Mean: {mean_val:.1f}')
        ax[j,i].axvline(median_val, color='blue', linestyle=':', linewidth=2, 
                        label=f'Median: {median_val:.1f}')

        # === 标注文本信息 ===
        # 1. 标明这是哪个光度-颜色区间 (使用 transAxes 定位，0.5代表居中)
        ax[j,i].text(0.5, 0.8, f'{Mr_thr_text[j]}, {Co_thr_text[j,i]}', 
                     transform=ax[j,i].transAxes, fontsize=12, ha='center', weight='bold')

        # 2. 标明 Ngal in fine bins (放在左侧居中位置)
        ax[j,i].text(0.4, 0.2, r'$N_{gal}^{fine-bin}$ = ' + f'{len(Ng8_arr)}', 
                     transform=ax[j,i].transAxes, fontsize=12, ha='left')
        
        # 3. 添加图例，显示 Mean 和 Median 的数值 (放在右侧)
        ax[j,i].legend(loc='center right', fontsize=10, frameon=False)

        # 设置坐标轴标签和范围
        ax[3,i].set_xlabel(r'$N_g^8$', fontsize=14)
        ax[j,0].set_ylabel(r'$P(N_g^8)$', fontsize=14)
        
        ax[j,i].set_xlim(55, 225)
        ax[j,i].set_ylim(0, 0.05)

plt.suptitle(r'$N_g^8$ Distribution (DDP defined) with Mean & Median, top 20%, ' + var + '_' + quad, y=0.92, fontsize=16, weight='bold')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 读取保存好的 npz 数据
# 请确保文件名与你上一步保存的一致
saved_data = np.load('./CCMD_' + var + '_' + quad + '_Ng8_data_DDP_test.npz')

# 2. 创建 4x4 画布
fig, ax = plt.subplots(4, 4, figsize=(20, 12), sharey=True, sharex=True)
plt.subplots_adjust(hspace=0, wspace=0)

for j in range(4): # 对应 Mr bins
    for i in range(4): # 对应 Color bins
        
        # 构造读取字典的 key
        key_suffix = M_thr_list[j] + '_C_' + C_thr_list[j,i]
        
        # 提取当前 panel 的 Ng8 数组和 top 20% 的 fsat
        Ng8_arr = saved_data['Ng8_thr_valid_' + key_suffix]
        
        # === 核心计算：均值和中位数 ===
        mean_val = np.mean(Ng8_arr)
        median_val = np.median(Ng8_arr)

        # 绘制直方图 (加上边缘线会让柱子更清晰)
        n_bins = int(np.ceil(Ng8_arr.max() - Ng8_arr.min()))
        ax[j,i].hist(Ng8_arr, bins=max(1, n_bins), density=True, 
                     color='lightgray', edgecolor='gray', alpha=0.7)
        # ======================================================
        
        # === 新增：绘制均值和中位数竖线 ===
        ax[j,i].axvline(mean_val, color='red', linestyle='--', linewidth=2, 
                        label=f'Mean: {mean_val:.1f}')
        ax[j,i].axvline(median_val, color='blue', linestyle=':', linewidth=2, 
                        label=f'Median: {median_val:.1f}')

        # === 标注文本信息 ===
        # 1. 标明这是哪个光度-颜色区间 (使用 transAxes 定位，0.5代表居中)
        ax[j,i].text(0.5, 0.8, f'{Mr_thr_text[j]}, {Co_thr_text[j,i]}', 
                     transform=ax[j,i].transAxes, fontsize=12, ha='center', weight='bold')
        
        # 2. 标明 Ngal in fine bins (放在左侧居中位置)
        ax[j,i].text(0.4, 0.2, r'$N_{gal}^{fine-bin}$ = ' + f'{len(Ng8_arr)}', 
                     transform=ax[j,i].transAxes, fontsize=12, ha='left')
        
        # 3. 添加图例，显示 Mean 和 Median 的数值 (放在右侧)
        ax[j,i].legend(loc='center right', fontsize=10, frameon=False)

        # 设置坐标轴标签和范围
        ax[3,i].set_xlabel(r'$N_g^8$', fontsize=14)
        ax[j,0].set_ylabel(r'$P(N_g^8)$', fontsize=14)
        
        ax[j,i].set_xlim(0, 225)
        ax[j,i].set_ylim(0, 0.05)

plt.suptitle(r'$N_g^8$ Distribution (DDP defined) with Mean & Median', y=0.92, fontsize=16, weight='bold')
plt.show()